In [ ]:
%pip install --upgrade crewai crewai-tools langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic
%pip install litellm
%pip install crewai-tools

from crewai.tools import BaseTool
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
# Approach 1


from crewai_tools import PDFSearchTool

class ResearchPDFSearchTool(PDFSearchTool):
    def __init__(self, pdf_path: str):
        super().__init__(
            pdf=pdf_path,
            config={
                "llm": {
                    "provider": "google",
                    "config": {
                        "model": "gemini/gemini-2.0-flash",
                        "temperature": 0,
                    },
                },
                "embedder": {
                    "provider": "google",
                    "config": {
                        "model": "models/text-embedding-004",
                    },
                },
            },
        )

pdf_search_tool = ResearchPDFSearchTool(
    "PDF_Folder/Calorie_ScienceDaily.pdf"
)

retriever_agent = Agent(
    role="PDF Researcher",
    goal="Answer questions using the supplied PDF document",
    backstory="You retrieve and summarize information accurately from PDF documents.",
    tools=[pdf_search_tool],
    llm=llm,
    verbose=True,
)

In [ ]:
# Approach 2




class ExplicitPDFSearchTool(BaseTool):
    name: str = "pdf_search"
    description: str = "Search indexed PDF content."

    def __init__(self, docs):
        super().__init__()

        self.embedding_model = GoogleGenerativeAIEmbeddings(
            model="models/gemini-embedding-001",
            google_api_key=GEMINI_API_KEY,
        )

        print("Creating document embeddings...")
        self.vectorstore = FAISS.from_documents(
            docs,
            self.embedding_model,
        )

    def _run(self, query: str) -> str:
        print(f"Creating query embedding for: {query}")

        documents = self.vectorstore.similarity_search(
            query,
            k=3,
        )

        print(f"Retrieved {len(documents)} document chunks.")

        return "\n\n".join(
            document.page_content
            for document in documents
        )

pdf_tool = ExplicitPDFSearchTool(docs)

agent = Agent(
    role="PDF Researcher",
    goal="Answer questions using the PDF",
    backstory="You retrieve information from indexed PDF documents.",
    tools=[pdf_tool],
    llm=llm,
)

crew = Crew(
    agents=[agent],
    tasks=[task],
    verbose=True,
)